# Decompose Default Experiment Into Canonical Stages

This notebook decomposes the default sklearn experiment into all canonical pipeline stages and validates stage outputs.

What this verifies:
- every canonical stage is present in the generated DVC stage plan
- stage ordering follows the canonical pipeline contract
- each stage exposes expected deps/outs/runtime metadata
- component stage groups (data, model, attack, detector, experiment) are visible for inspection

In [1]:
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

from deckard.experiment import ExperimentConfig
from deckard.experiment.canon import (
    CANONICAL_EXPERIMENT_PIPELINE_STAGES,
    CANONICAL_EXPERIMENT_STAGE_OUTPUT_KEYS,
)
from deckard.experiment.dvc import build_dvc_stage_plan

/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "deckard").exists() and (p / "examples").exists()), cwd)

BUILD_DIR = PROJECT_ROOT / "docs" / "build" / "dvc_notebook"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"BUILD_DIR={BUILD_DIR}")

PROJECT_ROOT=/Users/c.meyers/Documents/deckard
BUILD_DIR=/Users/c.meyers/Documents/deckard/docs/build/dvc_notebook


In [3]:
config_dir = (PROJECT_ROOT / "examples" / "sklearn" / "config").as_posix()

with initialize_config_dir(version_base="1.3", config_dir=config_dir):
    cfg = compose(
        config_name="default",
        overrides=[
            "+stage=persist",
            "files=default",
            f"+files.params_file={BUILD_DIR.as_posix()}/params.yaml",
            f"+files.score_file={BUILD_DIR.as_posix()}/scores.json",
            f"+files.log_file={BUILD_DIR.as_posix()}/run.log",
            f"+files.error_file={BUILD_DIR.as_posix()}/error.log",
            "hydra/job_logging=none",
            "hydra/hydra_logging=none",
        ],
    )

resolved = OmegaConf.to_container(cfg, resolve=True)
allowed = set(ExperimentConfig.__dataclass_fields__.keys())
payload = {"_target_": "deckard.ExperimentConfig"}
for key, value in resolved.items():
    if key in allowed:
        payload[key] = value

experiment = instantiate(payload)
print(type(experiment).__name__)

ExperimentConfig


In [4]:
print("Canonical Pipeline Stage Order:")
for idx, stage in enumerate(CANONICAL_EXPERIMENT_PIPELINE_STAGES, start=1):
    print(f"{idx:02d}. {stage}")

print("\nCanon Stage Output Contract (keys):")
for stage, keys in CANONICAL_EXPERIMENT_STAGE_OUTPUT_KEYS.items():
    print(f"- {stage}: {list(keys)}")

Canonical Pipeline Stage Order:
01. load
02. sample
03. pipeline
04. data_score
05. data_persist
06. apply_fit_defense
07. train
08. apply_predict_defense
09. model_score
10. model_persist
11. generation
12. attack_score
13. attack_persist
14. detector-train
15. detector-defense
16. detector_score
17. detector_persist
18. score
19. persist

Canon Stage Output Contract (keys):
- load: ['data_file', 'metadata_file', 'post_pipeline_data_file', 'post_sample_data_file']
- sample: ['data_file', 'metadata_file', 'post_pipeline_data_file', 'post_sample_data_file']
- pipeline: ['data_file', 'metadata_file', 'post_pipeline_data_file', 'post_sample_data_file']
- data-persist: ['data_file', 'metadata_file', 'post_pipeline_data_file', 'post_sample_data_file']
- train: ['model_file', 'test_predictions_file', 'train_predictions_file', 'training_predictions_file', 'test_probabilities_file', 'training_probabilities_file']
- model-persist: ['model_file', 'test_predictions_file', 'train_predictions_file'

In [5]:
stage_plan = build_dvc_stage_plan(experiment, mode="single")

print(f"Generated stage count: {len(stage_plan)}")
print("\nGenerated Stage Names (first 40):")
for entry in stage_plan[:40]:
    print(f"- {entry['name']} | stage={entry['stage']} | runtime={entry['runtime_stage']} | component={entry['component']}")
if len(stage_plan) > 40:
    print("...")

canonical_norm = [stage.replace("_", "-") for stage in CANONICAL_EXPERIMENT_PIPELINE_STAGES]


def to_canonical_base(stage_token: str) -> str:
    token = stage_token.replace("_", "-")
    for base in sorted(canonical_norm, key=len, reverse=True):
        if token == base or token.startswith(f"{base}-"):
            return base
    return token


def is_required(base_stage: str) -> bool:
    if base_stage.startswith("detector-"):
        return getattr(experiment, "detector", None) is not None
    if base_stage.startswith("attack-") or base_stage == "generation":
        return getattr(experiment, "attack", None) is not None or len(getattr(experiment, "_attack_chain", []) or []) > 0
    if base_stage.startswith("model-") or base_stage == "train":
        return getattr(experiment, "model", None) is not None
    if base_stage.startswith("apply-"):
        return getattr(experiment, "defense", None) is not None
    if base_stage.startswith("data-"):
        return getattr(experiment, "data", None) is not None
    return True

present_norm = {to_canonical_base(entry["stage"]) for entry in stage_plan}
required_norm = {stage for stage in canonical_norm if is_required(stage)}
missing_required = sorted(required_norm - present_norm)
missing_optional = sorted((set(canonical_norm) - required_norm) - present_norm)
extra = sorted(present_norm - set(canonical_norm))

print("\nCoverage Check (normalized canonical bases):")
print(f"missing_required={missing_required}")
print(f"missing_optional={missing_optional}")
print(f"extra={extra}")

stage_names = [entry["name"] for entry in stage_plan]
assert any(name.startswith("data__pipeline") for name in stage_names), "Expected decomposed data__pipeline stage"
assert not missing_required, f"Missing required canonical stages: {missing_required}"

Generated stage count: 15

Generated Stage Names (first 40):
- data__load | stage=load | runtime=load | component=data
- data__sample | stage=sample | runtime=sample | component=data
- data__pipeline | stage=pipeline | runtime=pipeline | component=data
- data__data-score | stage=data-score | runtime=score | component=data
- data__data-persist | stage=data-persist | runtime=persist | component=data
- defense__apply-fit-defense-class-labels | stage=apply-fit-defense-class-labels | runtime=defense | component=defense
- model__train | stage=train | runtime=train | component=model
- defense__apply-predict-defense-class-labels | stage=apply-predict-defense-class-labels | runtime=defense | component=defense
- model__model-score | stage=model-score | runtime=score | component=model
- model__model-persist | stage=model-persist | runtime=persist | component=model
- attack__generation-hsj | stage=generation-hsj | runtime=attack | component=attack
- attack__attack-score | stage=attack-score | runt

In [6]:
print("Per-stage dependency/output summary:\n")
for entry in stage_plan:
    print(entry["name"])
    print(f"  deps: {len(entry.get('deps', []))}")
    print(f"  outs: {len(entry.get('outs', []))}")
    print(f"  metrics: {len(entry.get('metrics', []))}")
    print(f"  plots: {len(entry.get('plots', []))}")
    if entry.get("runtime_overrides"):
        print(f"  runtime_overrides: {entry['runtime_overrides']}")
    print()

Per-stage dependency/output summary:

data__load
  deps: 3
  outs: 0
  metrics: 0
  plots: 0

data__sample
  deps: 3
  outs: 0
  metrics: 0
  plots: 0

data__pipeline
  deps: 4
  outs: 0
  metrics: 0
  plots: 0

data__data-score
  deps: 3
  outs: 0
  metrics: 0
  plots: 0
  runtime_overrides: ['score.scope=data']

data__data-persist
  deps: 4
  outs: 1
  metrics: 0
  plots: 0
  runtime_overrides: ['persist.scope=data']

defense__apply-fit-defense-class-labels
  deps: 4
  outs: 0
  metrics: 0
  plots: 0
  runtime_overrides: ['defense.apply=fit', 'stage_alias=class-labels']

model__train
  deps: 4
  outs: 0
  metrics: 0
  plots: 0

defense__apply-predict-defense-class-labels
  deps: 4
  outs: 0
  metrics: 0
  plots: 0
  runtime_overrides: ['defense.apply=predict', 'stage_alias=class-labels']

model__model-score
  deps: 3
  outs: 0
  metrics: 0
  plots: 0
  runtime_overrides: ['score.scope=model']

model__model-persist
  deps: 4
  outs: 1
  metrics: 0
  plots: 0
  runtime_overrides: ['per

## Verification Notes

If all assertions pass, the default experiment has been decomposed into every canonical stage currently defined by the canon pipeline contract.

Use the printed per-stage summary to validate:
- ordering
- component routing
- output/dependency volume
- stage-specific runtime overrides